# Baby Category — Temporal Market Basket Analysis
## Biological Purchase Clock & Lifecycle Phase Detection

**Objective:** Identify the *Biological Purchase Clock* by analysing how subclass preferences shift relative to each shopper's first observed baby purchase (T-Zero).

**What this notebook delivers:**
1. **T-Zero Alignment** — Every shopper anchored to Day 0 (their first baby purchase)
2. **Subclass Velocity** — Mean day of first purchase per subclass across the cohort
3. **Cohort Market Basket Analysis** — Association rules sliced into 3-month windows (0-3m, 4-6m, …)
4. **Ridge Plot** — Subclass purchase distributions across the first 24 months
5. **Transition Heatmap** — Probability of moving from Subclass A → B over time
6. **Lifecycle Phase Summary** — 4-5 dominant phases derived from subclass clusters

**Inputs (Phase 1 outputs):**
- `combined_shoppers.parquet` — raw transaction data
- `baby_shopper_profiles_phase1.parquet` *(optional)* — segment labels from Phase 1

---

## 1. Setup & Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# ── Visual theme (consistent with Phase 1) ────────────────────────────────────
PALETTE = ['#2C7BB6', '#D7191C', '#1A9641', '#FDAE61', '#762A83',
           '#F46D43', '#4DAC26', '#B2ABD2', '#E66101', '#5E3C99']
sns.set_theme(style='whitegrid', palette=PALETTE)
plt.rcParams.update({
    'figure.dpi': 130,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'figure.titlesize': 13,
})
pd.set_option('display.max_columns', 60)
pd.set_option('display.float_format', '{:.3f}'.format)

# ── Constants ─────────────────────────────────────────────────────────────────
ANALYSIS_HORIZON_MONTHS = 24     # biological clock window
COHORT_SIZE_MONTHS      = 3      # basket cohort width (0-3m, 4-6m …)
MIN_SUPPORT             = 0.02   # MBA minimum support
MIN_CONFIDENCE          = 0.20   # MBA minimum confidence
MIN_LIFT                = 1.0    # MBA minimum lift
TOP_N_SUBCLASS          = 20     # subclasses shown in ridge/heatmap
RANDOM_STATE            = 42

# Cohort labels & boundaries (months since T-Zero)
COHORT_EDGES  = list(range(0, ANALYSIS_HORIZON_MONTHS + 1, COHORT_SIZE_MONTHS))
COHORT_LABELS = [f"{COHORT_EDGES[i]}-{COHORT_EDGES[i+1]}m"
                 for i in range(len(COHORT_EDGES) - 1)]

print("Setup complete.")
print(f"Cohort windows: {COHORT_LABELS}")

## 2. Data Loading & Cleaning

In [ ]:
# ── 2.1 Load raw transactions ─────────────────────────────────────────────────
raw = pd.read_parquet(r"C:\Users\dacdatalabs.ojt2\Desktop\workspace\DS\datasets\combined_shoppers.parquet")
print(f"Raw shape : {raw.shape}")
print(f"Columns   : {list(raw.columns)}")
raw.head(3)

In [ ]:
# ── 2.2 Normalise & parse ─────────────────────────────────────────────────────
raw["header_tran_date"] = raw["header_tran_date"].str.replace("Sept", "Sep", case=False)

for col in ['department_cleaned', 'subclass_cleaned', 'subclass']:
    if col in raw.columns:
        raw[col] = raw[col].str.lower().str.strip()

raw['date'] = pd.to_datetime(raw['header_tran_date'], format='%d %b %Y', errors='coerce')
n_bad = raw['date'].isna().sum()
print(f"Unparseable dates removed : {n_bad:,}")
raw = raw.dropna(subset=['date', 'gcr_persistent_id'])

# ── 2.3 Remove promo / clearance rows ────────────────────────────────────────
PROMO_VALS = {'md promo', 'iw promotions', 'iaf clearance', 'iaf promotions'}
before = len(raw)
raw = raw[~raw['subclass'].isin(PROMO_VALS)]
print(f"Promo rows removed        : {before - len(raw):,}")

# ── 2.4 Rename for clarity ────────────────────────────────────────────────────
df = raw.rename(columns={'gcr_persistent_id': 'shopper_id'}).copy()
df = df.sort_values(['shopper_id', 'date']).reset_index(drop=True)

print(f"\nWorking dataset: {df.shape[0]:,} rows | {df['shopper_id'].nunique():,} shoppers")
print(f"Unique subclasses: {df['subclass'].nunique()}")
print(f"Date range: {df['date'].min().date()} → {df['date'].max().date()}")

---
## 3. Establish T-Zero — The Biological Clock Entry Point

**T-Zero** is each shopper's first ever observed baby-category transaction.  
All subsequent analysis is expressed as **days since T-Zero** (relative time), removing any calendar-date confound and aligning all shoppers to a common biological timeline.

In [ ]:
def establish_t_zero(transactions: pd.DataFrame,
                     shopper_col: str = 'shopper_id',
                     date_col: str = 'date') -> pd.DataFrame:
    """
    Identify each shopper's first transaction date (T-Zero / Day 0)
    and compute relative days from that anchor.

    Parameters
    ----------
    transactions : pd.DataFrame
        Must contain shopper_col and date_col.
    shopper_col  : identifier column
    date_col     : transaction date column (datetime)

    Returns
    -------
    pd.DataFrame with two new columns:
        t_zero          — anchor date for each shopper
        days_since_zero — signed integer days relative to T-Zero
    """
    t_zero_map = (
        transactions
        .groupby(shopper_col)[date_col]
        .min()
        .rename('t_zero')
    )
    out = transactions.join(t_zero_map, on=shopper_col)
    out['days_since_zero'] = (out[date_col] - out['t_zero']).dt.days
    return out


df = establish_t_zero(df)

# Filter to the biological analysis window (0 – 24 months)
MAX_DAYS = ANALYSIS_HORIZON_MONTHS * 30
df_window = df[(df['days_since_zero'] >= 0) & (df['days_since_zero'] <= MAX_DAYS)].copy()

# Compute relative month bucket (0-indexed)
df_window['month_since_zero'] = (df_window['days_since_zero'] // 30).clip(upper=ANALYSIS_HORIZON_MONTHS - 1)

# Assign cohort label
df_window['cohort'] = pd.cut(
    df_window['month_since_zero'],
    bins=COHORT_EDGES,
    labels=COHORT_LABELS,
    right=False,
    include_lowest=True
)

print(f"Transactions within {ANALYSIS_HORIZON_MONTHS}-month window : {len(df_window):,}")
print(f"Shoppers with data in window : {df_window['shopper_id'].nunique():,}")
print(f"\nCohort distribution (shoppers with ≥1 transaction):")
print(df_window.groupby('cohort', observed=True)['shopper_id'].nunique().to_string())

In [ ]:
# ── T-Zero validation: entry day should be 0 for all shoppers ────────────────
tz_check = df_window.groupby('shopper_id')['days_since_zero'].min()
assert (tz_check == 0).all(), "T-Zero mismatch — some shoppers do not start at Day 0!"
print("✓ T-Zero validation passed: all shoppers anchored to Day 0.")

# ── Quick peek at the aligned data ───────────────────────────────────────────
df_window[['shopper_id', 'date', 't_zero', 'days_since_zero',
           'month_since_zero', 'cohort', 'subclass']].head(12)

---
## 4. Subclass Velocity — The Purchase Timeline

**Subclass Velocity** answers: *"On average, at what day relative to T-Zero does a shopper first buy a given subclass?"*

A low mean day = shoppers buy this early in the journey (e.g., Newborn Nappies).  
A high mean day = shoppers buy this later (e.g., Potty Chairs, Toddler Cups).

In [ ]:
# ── 4.1 First purchase day per shopper × subclass ────────────────────────────
first_touch = (
    df_window
    .groupby(['shopper_id', 'subclass'], observed=True)['days_since_zero']
    .min()
    .reset_index(name='first_day')
)

# ── 4.2 Subclass velocity statistics ─────────────────────────────────────────
velocity = (
    first_touch
    .groupby('subclass')['first_day']
    .agg(
        mean_day   = 'mean',
        median_day = 'median',
        std_day    = 'std',
        n_shoppers = 'count',
    )
    .reset_index()
    .sort_values('mean_day')
)

# Month equivalent for readability
velocity['mean_month'] = (velocity['mean_day'] / 30).round(1)

# Filter to subclasses bought by enough shoppers (signal quality)
MIN_SHOPPERS = 50
velocity = velocity[velocity['n_shoppers'] >= MIN_SHOPPERS]

print(f"Subclasses with ≥{MIN_SHOPPERS} shoppers : {len(velocity)}")
print("\n=== Earliest Subclasses (Mean Day of First Purchase) ===")
print(velocity.head(10).to_string(index=False))
print("\n=== Latest Subclasses ===")
print(velocity.tail(10).to_string(index=False))

In [ ]:
# ── 4.3 Visualise subclass velocity ──────────────────────────────────────────
top_v = velocity.head(TOP_N_SUBCLASS)

fig, ax = plt.subplots(figsize=(13, 7))
colors = [PALETTE[0] if m <= 6 else PALETTE[2] if m <= 12 else PALETTE[1]
          for m in top_v['mean_month']]

bars = ax.barh(top_v['subclass'], top_v['mean_day'], color=colors,
               edgecolor='white', linewidth=0.4)

# Error bars (±1 std)
ax.errorbar(top_v['mean_day'], range(len(top_v)),
            xerr=top_v['std_day'].fillna(0), fmt='none',
            ecolor='grey', elinewidth=0.8, capsize=2)

# Annotate with month label
for bar, (_, row) in zip(bars, top_v.iterrows()):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height() / 2,
            f"mo {row['mean_month']:.1f}", va='center', fontsize=7.5)

# Phase bands
for start, end, label, alpha in [
    (0, 90,  '0-3m\nNewborn',    0.07),
    (90, 180, '3-6m\nInfant',    0.07),
    (180, 365,'6-12m\nGrowth',   0.07),
    (365, 720,'12-24m\nToddler', 0.07),
]:
    ax.axvspan(start, min(end, top_v['mean_day'].max() + 50),
               alpha=alpha, color='grey', zorder=0)
    ax.text((start + end) / 2, len(top_v) - 0.3, label,
            ha='center', fontsize=7, color='#444')

legend_patches = [
    mpatches.Patch(color=PALETTE[0], label='Early (≤6m)'),
    mpatches.Patch(color=PALETTE[2], label='Mid (7-12m)'),
    mpatches.Patch(color=PALETTE[1], label='Late (>12m)'),
]
ax.legend(handles=legend_patches, fontsize=9, loc='lower right')

ax.set_xlabel('Mean Day of First Purchase (relative to T-Zero)')
ax.set_title('Subclass Velocity — When Does Each Category Enter the Baby Journey?',
             fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('P2_01_subclass_velocity.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Ridge Plot — Subclass Purchase Distributions over 24 Months

A **Ridge Plot** (joy plot) stacks the KDE of purchase-day distributions for each subclass, revealing how categories peak and trail off at different points in the baby lifecycle.

- Sharp early peaks → newborn essentials (nappies, feeding)
- Broad later peaks → developmental items (toys, furniture)
- Bimodal → replenishment items (wipes, formula)

In [ ]:
from scipy.stats import gaussian_kde

# ── 5.1 Select top subclasses by shopper count for ridge plot ─────────────────
top_subclasses = (
    df_window.groupby('subclass', observed=True)['shopper_id']
    .nunique()
    .sort_values(ascending=False)
    .head(TOP_N_SUBCLASS)
    .index.tolist()
)

# Order by mean purchase day (velocity) — earliest at top
velocity_ordered = velocity[velocity['subclass'].isin(top_subclasses)].sort_values('mean_day')
ordered_labels   = velocity_ordered['subclass'].tolist()

# ── 5.2 KDE grid ──────────────────────────────────────────────────────────────
x_grid = np.linspace(0, MAX_DAYS, 500)
kde_results = {}
for sc in ordered_labels:
    vals = df_window[df_window['subclass'] == sc]['days_since_zero'].dropna().values
    if len(vals) < 5:
        continue
    try:
        kde = gaussian_kde(vals, bw_method='silverman')
        kde_results[sc] = kde(x_grid)
    except Exception:
        pass

ordered_labels = [sc for sc in ordered_labels if sc in kde_results]
print(f"Ridge plot will show {len(ordered_labels)} subclasses.")

In [ ]:
# ── 5.3 Draw ridge plot ───────────────────────────────────────────────────────
N    = len(ordered_labels)
OVERLAP = 2.8          # vertical overlap factor
fig_h   = max(8, N * 0.7)

fig, ax = plt.subplots(figsize=(14, fig_h))

# Colour ramp: early = blue, late = red
cmap    = plt.cm.RdYlBu_r
mean_days = [velocity_ordered[velocity_ordered['subclass'] == sc]['mean_day'].values[0]
             if sc in velocity_ordered['subclass'].values else MAX_DAYS / 2
             for sc in ordered_labels]
norm_md = MinMaxScaler().fit_transform(np.array(mean_days).reshape(-1, 1)).flatten()

for i, sc in enumerate(ordered_labels):
    density   = kde_results[sc]
    y_base    = N - i            # baseline for this ridge
    y_scaled  = density / density.max() * OVERLAP + y_base
    color     = cmap(norm_md[i])

    ax.fill_between(x_grid / 30, y_scaled, y_base,
                    color=color, alpha=0.65)
    ax.plot(x_grid / 30, y_scaled, color=color, linewidth=0.8)

    # Subclass label on the left
    ax.text(-0.3, y_base + 0.15, sc, ha='right', va='bottom',
            fontsize=7.5, color='#333')

# Phase lines
for month, label in [(3, '3m'), (6, '6m'), (12, '12m'), (18, '18m')]:
    ax.axvline(month, color='#aaa', linewidth=0.7, linestyle='--', zorder=0)
    ax.text(month, N + OVERLAP * 0.6, label, ha='center',
            fontsize=8, color='#666')

sm = plt.cm.ScalarMappable(cmap=cmap,
     norm=plt.Normalize(vmin=0, vmax=MAX_DAYS / 30))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, orientation='vertical',
                    fraction=0.02, pad=0.01)
cbar.set_label('Mean Month of First Purchase', fontsize=9)

ax.set_xlabel('Months Since T-Zero (First Baby Purchase)', fontsize=10)
ax.set_xlim(-0.5, ANALYSIS_HORIZON_MONTHS)
ax.set_yticks([])
ax.set_title('Ridge Plot — Baby Subclass Purchase Distributions over First 24 Months',
             fontweight='bold', fontsize=13)
ax.spines[['top', 'right', 'left']].set_visible(False)
plt.tight_layout()
plt.savefig('P2_02_ridge_plot.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Cohort Market Basket Analysis

Market basket rules are computed **separately per 3-month cohort**, answering:  
*"What do shoppers buy together during months 0-3 vs 4-6 vs … vs 22-24?"*

This reveals how rules like `{Diapers} → {Wipes}` evolve into `{Finger Foods} → {Training Pants}` as the baby grows.

In [ ]:
def build_basket_for_cohort(cohort_df: pd.DataFrame,
                             shopper_col: str = 'shopper_id',
                             subclass_col: str = 'subclass') -> pd.DataFrame:
    """
    Given transactions within a cohort window, build a one-hot
    shopper × subclass matrix and return it.
    """
    basket = (
        cohort_df
        .groupby([shopper_col, subclass_col], observed=True)
        .size()
        .unstack(fill_value=0)
        .clip(upper=1)          # binary (bought / not bought)
        .astype(bool)
    )
    return basket


all_rules = []

for cohort_label in COHORT_LABELS:
    cohort_data = df_window[df_window['cohort'] == cohort_label]
    n_shoppers  = cohort_data['shopper_id'].nunique()

    if n_shoppers < 30:
        print(f"  ⚠ {cohort_label}: too few shoppers ({n_shoppers}) — skipped.")
        continue

    basket = build_basket_for_cohort(cohort_data)

    # Apriori frequent itemsets
    try:
        freq_items = apriori(basket,
                             min_support=MIN_SUPPORT,
                             use_colnames=True,
                             max_len=2)

        if freq_items.empty:
            print(f"  ⚠ {cohort_label}: no frequent itemsets at support={MIN_SUPPORT}")
            continue

        rules = association_rules(freq_items,
                                  metric='confidence',
                                  min_threshold=MIN_CONFIDENCE)
        rules = rules[rules['lift'] >= MIN_LIFT]
        rules['cohort']     = cohort_label
        rules['n_shoppers'] = n_shoppers
        all_rules.append(rules)
        print(f"  ✓ {cohort_label}: {n_shoppers:,} shoppers | "
              f"{len(freq_items)} itemsets | {len(rules)} rules")

    except Exception as e:
        print(f"  ✗ {cohort_label}: error — {e}")

rules_df = pd.concat(all_rules, ignore_index=True) if all_rules else pd.DataFrame()
print(f"\nTotal rules across all cohorts: {len(rules_df):,}")

In [ ]:
# ── 6.1 Format antecedents / consequents as readable strings ──────────────────
if not rules_df.empty:
    rules_df['antecedents_str'] = rules_df['antecedents'].apply(
        lambda x: ', '.join(sorted(x)))
    rules_df['consequents_str'] = rules_df['consequents'].apply(
        lambda x: ', '.join(sorted(x)))
    rules_df['rule'] = rules_df['antecedents_str'] + ' → ' + rules_df['consequents_str']

    print("=== Top 10 Rules by Lift (all cohorts) ===")
    print(
        rules_df[['cohort', 'rule', 'support', 'confidence', 'lift']]
        .sort_values('lift', ascending=False)
        .head(10)
        .to_string(index=False)
    )

In [ ]:
# ── 6.2 Visualise: top rules per cohort by lift ───────────────────────────────
if not rules_df.empty:
    TOP_RULES_PER_COHORT = 5
    top_by_cohort = (
        rules_df
        .sort_values(['cohort', 'lift'], ascending=[True, False])
        .groupby('cohort', observed=True)
        .head(TOP_RULES_PER_COHORT)
    )

    cohorts_present = top_by_cohort['cohort'].unique()
    n_cols = 2
    n_rows = (len(cohorts_present) + 1) // n_cols

    fig, axes = plt.subplots(n_rows, n_cols,
                             figsize=(16, n_rows * 3.5),
                             squeeze=False)
    fig.suptitle('Top Association Rules per Cohort Window (by Lift)',
                 fontsize=13, fontweight='bold')

    for idx, cohort in enumerate(cohorts_present):
        ax  = axes[idx // n_cols][idx % n_cols]
        sub = top_by_cohort[top_by_cohort['cohort'] == cohort]

        bars = ax.barh(sub['rule'], sub['lift'],
                       color=PALETTE[idx % len(PALETTE)], edgecolor='white')
        for bar, conf in zip(bars, sub['confidence']):
            ax.text(bar.get_width() + 0.02,
                    bar.get_y() + bar.get_height() / 2,
                    f"conf={conf:.2f}",
                    va='center', fontsize=7)

        ax.set_title(f"Cohort {cohort}  (n={sub['n_shoppers'].iloc[0]:,} shoppers)",
                     fontsize=10)
        ax.set_xlabel('Lift')
        ax.invert_yaxis()
        ax.tick_params(axis='y', labelsize=7.5)
        ax.axvline(1.0, color='grey', linewidth=0.7, linestyle='--')

    # Hide unused subplots
    for idx in range(len(cohorts_present), n_rows * n_cols):
        axes[idx // n_cols][idx % n_cols].set_visible(False)

    plt.tight_layout()
    plt.savefig('P2_03_cohort_mba_rules.png', dpi=150, bbox_inches='tight')
    plt.show()

In [ ]:
# ── 6.3 Rule evolution: track how a key antecedent's top rule changes ─────────
if not rules_df.empty:
    # Identify the single most common antecedent across all cohorts
    key_ant = rules_df['antecedents_str'].value_counts().idxmax()

    evolution = (
        rules_df[rules_df['antecedents_str'] == key_ant]
        .sort_values(['cohort', 'lift'], ascending=[True, False])
        .groupby('cohort', observed=True)
        .first()
        .reset_index()[['cohort', 'rule', 'support', 'confidence', 'lift']]
    )

    print(f"=== Rule Evolution for antecedent: '{key_ant}' ===")
    print(evolution.to_string(index=False))

---
## 7. Transition Heatmap — Subclass-to-Subclass Movement Over Time

The **Transition Heatmap** shows, for each pair of consecutive purchases within the 24-month window, the conditional probability that a shopper who just bought Subclass A will next purchase Subclass B.

Unlike Phase 1's global transition matrix, this version is computed **per cohort** to capture how transition probabilities shift as the baby grows.

In [ ]:
# ── 7.1 Build global transition matrix (all cohorts combined) ─────────────────
df_sorted = df_window.sort_values(['shopper_id', 'days_since_zero'])
df_sorted['next_subclass'] = df_sorted.groupby('shopper_id')['subclass'].shift(-1)

# Only same-shopper transitions (shift introduces cross-shopper leakage at group boundaries)
df_sorted['_next_shopper'] = df_sorted.groupby('shopper_id')['shopper_id'].shift(-1)
transitions_raw = df_sorted[
    df_sorted['next_subclass'].notna() &
    (df_sorted['shopper_id'] == df_sorted['_next_shopper'])
].copy()

# Restrict to top N subclasses
hm_subclasses = top_subclasses[:min(TOP_N_SUBCLASS, 15)]
trans_filtered = transitions_raw[
    transitions_raw['subclass'].isin(hm_subclasses) &
    transitions_raw['next_subclass'].isin(hm_subclasses)
]

# Count & normalise
trans_counts = (
    trans_filtered
    .groupby(['subclass', 'next_subclass'], observed=True)
    .size()
    .reset_index(name='count')
)
row_totals = trans_counts.groupby('subclass')['count'].sum().rename('row_total')
trans_counts = trans_counts.join(row_totals, on='subclass')
trans_counts['prob'] = trans_counts['count'] / trans_counts['row_total']

hm_pivot = (
    trans_counts
    .pivot(index='subclass', columns='next_subclass', values='prob')
    .reindex(index=hm_subclasses, columns=hm_subclasses)
    .fillna(0)
)

print(f"Transition matrix: {hm_pivot.shape}")

In [ ]:
# ── 7.2 Plot global transition heatmap ───────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 10))
sns.heatmap(
    hm_pivot,
    annot=True, fmt='.2f',
    cmap='YlOrRd',
    linewidths=0.4, linecolor='white',
    cbar_kws={'label': 'Transition Probability'},
    ax=ax,
    vmin=0, vmax=hm_pivot.values[hm_pivot.values < 1].max() if (hm_pivot.values < 1).any() else 1
)
ax.set_title(
    'Subclass Transition Probability Heatmap (First 24 Months)\n'
    'Row = Current Subclass  |  Column = Next Subclass Purchased',
    fontweight='bold', fontsize=12
)
ax.set_xlabel('Next Subclass', fontsize=10)
ax.set_ylabel('Current Subclass', fontsize=10)
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.savefig('P2_04_transition_heatmap_global.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── 7.3 Cohort-level transition heatmaps (Early vs Late) ─────────────────────
# Compare first cohort (0-3m) against a later cohort (e.g. 12-15m)
compare_cohorts = [COHORT_LABELS[0], COHORT_LABELS[4]] if len(COHORT_LABELS) > 4 \
                  else COHORT_LABELS[:2]

fig, axes = plt.subplots(1, 2, figsize=(20, 8))
fig.suptitle('Transition Heatmap Evolution: Early vs Later Cohort',
             fontsize=13, fontweight='bold')

for ax, cohort_label in zip(axes, compare_cohorts):
    sub = transitions_raw[transitions_raw['cohort'] == cohort_label]
    sub = sub[
        sub['subclass'].isin(hm_subclasses) &
        sub['next_subclass'].isin(hm_subclasses)
    ]

    if sub.empty:
        ax.set_title(f"{cohort_label} — no data")
        continue

    tc = sub.groupby(['subclass', 'next_subclass'], observed=True).size().reset_index(name='cnt')
    rt = tc.groupby('subclass')['cnt'].sum().rename('rt')
    tc = tc.join(rt, on='subclass')
    tc['prob'] = tc['cnt'] / tc['rt']

    pivot = (
        tc.pivot(index='subclass', columns='next_subclass', values='prob')
          .reindex(index=hm_subclasses, columns=hm_subclasses)
          .fillna(0)
    )

    sns.heatmap(pivot, annot=True, fmt='.2f', cmap='Blues',
                linewidths=0.3, linecolor='white',
                cbar_kws={'label': 'Probability'},
                ax=ax, vmin=0, vmax=0.6)
    ax.set_title(f"Cohort {cohort_label}  "
                 f"(n={sub['shopper_id'].nunique():,} shoppers)",
                 fontsize=10)
    ax.set_xlabel('Next Subclass', fontsize=8)
    ax.set_ylabel('Current Subclass', fontsize=8)
    ax.tick_params(axis='x', labelrotation=45, labelsize=7)
    ax.tick_params(axis='y', labelrotation=0,  labelsize=7)

plt.tight_layout()
plt.savefig('P2_05_transition_heatmap_cohort_compare.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 8. Lifecycle Phase Discovery

We identify distinct **Lifecycle Phases** by:
1. Computing each subclass's mean purchase month (velocity)
2. Clustering subclasses into groups based on temporal proximity and co-purchase patterns
3. Characterising each phase by its dominant subclasses and time window

The result is a data-driven taxonomy of the baby purchase journey.

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler

# ── 8.1 Build subclass feature matrix ────────────────────────────────────────
# Features: mean day, median day, std day, peak cohort (mode)
cohort_order = {c: i for i, c in enumerate(COHORT_LABELS)}

peak_cohort = (
    df_window.groupby(['subclass', 'cohort'], observed=True)
    .size()
    .reset_index(name='cnt')
    .sort_values('cnt', ascending=False)
    .groupby('subclass', observed=True)
    .first()['cohort']
    .map(cohort_order)
    .rename('peak_cohort_idx')
)

sc_features = (
    velocity
    .set_index('subclass')
    [['mean_day', 'median_day', 'std_day', 'n_shoppers']]
    .join(peak_cohort)
    .dropna()
)

# Normalise
scaler     = StandardScaler()
X_sc       = scaler.fit_transform(sc_features)

print(f"Subclass feature matrix: {sc_features.shape}")

In [ ]:
# ── 8.2 Hierarchical clustering (ward linkage) — n=5 phases ──────────────────
N_PHASES = 5
hc = AgglomerativeClustering(n_clusters=N_PHASES, linkage='ward')
sc_features = sc_features.copy()
sc_features['phase_cluster'] = hc.fit_predict(X_sc)

print("Subclasses per phase cluster:")
print(sc_features['phase_cluster'].value_counts().sort_index())

In [ ]:
# ── 8.3 Auto-name phases from centroid mean day ───────────────────────────────
phase_centroids = sc_features.groupby('phase_cluster')['mean_day'].median().sort_values()

PHASE_NAME_MAP = {
    0: "Phase 1 — Newborn Essentials   (0-3m)",
    1: "Phase 2 — Infant Foundations   (3-6m)",
    2: "Phase 3 — Developmental Growth (6-12m)",
    3: "Phase 4 — Active Exploration   (12-18m)",
    4: "Phase 5 — Toddler Transition   (18-24m)",
}

# Remap cluster IDs ordered by mean_day
rank_order = {old: new for new, old in enumerate(phase_centroids.index)}
sc_features['phase'] = sc_features['phase_cluster'].map(rank_order).map(PHASE_NAME_MAP)

print("\n=== Phase Cluster → Name Mapping ===")
for pc, name in sorted(PHASE_NAME_MAP.items()):
    members = sc_features[sc_features['phase'] == name].index.tolist()
    mean_d  = sc_features[sc_features['phase'] == name]['mean_day'].median()
    print(f"  {name}  |  median day {mean_d:.0f}  |  {members}")

In [ ]:
# ── 8.4 Visualise: subclasses colour-coded by lifecycle phase ─────────────────
sc_plot = sc_features.reset_index().sort_values('mean_day')
phase_list = [p for p in PHASE_NAME_MAP.values() if p in sc_plot['phase'].values]
phase_colors = {name: PALETTE[i % len(PALETTE)] for i, name in enumerate(PHASE_NAME_MAP.values())}

fig, ax = plt.subplots(figsize=(13, max(6, len(sc_plot) * 0.45)))

for _, row in sc_plot.iterrows():
    color = phase_colors.get(row['phase'], '#aaa')
    ax.barh(row['subclass'], row['mean_day'],
            color=color, edgecolor='white', linewidth=0.3)
    ax.errorbar(row['mean_day'], row['subclass'],
                xerr=row['std_day'], fmt='none',
                ecolor='grey', elinewidth=0.6, capsize=2)

legend_patches = [mpatches.Patch(color=phase_colors[p], label=p.strip())
                  for p in PHASE_NAME_MAP.values()
                  if p in sc_plot['phase'].values]
ax.legend(handles=legend_patches, fontsize=8,
          loc='lower right', framealpha=0.9)

# Phase boundary lines
for day, label in [(90, '3m'), (180, '6m'), (365, '12m'), (540, '18m')]:
    ax.axvline(day, color='#888', linewidth=0.8, linestyle='--')
    ax.text(day + 3, 0, label, fontsize=8, color='#555', va='bottom')

ax.set_xlabel('Mean Day of First Purchase (relative to T-Zero)', fontsize=10)
ax.set_title('Baby Subclass Lifecycle Phases — Colour-Coded by Cluster',
             fontweight='bold')
ax.invert_yaxis()
plt.tight_layout()
plt.savefig('P2_06_phase_subclass_chart.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Lifecycle Phase Summary Table

The final deliverable: a human-readable summary of the **4-5 distinct Lifecycle Phases** with their:
- Time window
- Dominant subclasses
- Shopper penetration
- Key basket rules active in that phase
- Strategic interpretation

In [ ]:
# ── 9.1 Phase-level aggregation ───────────────────────────────────────────────
phase_summary_rows = []

for phase_name in PHASE_NAME_MAP.values():
    phase_subs = sc_features[sc_features['phase'] == phase_name]
    if phase_subs.empty:
        continue

    # Subclasses in this phase
    phase_sub_list = phase_subs.sort_values('mean_day').index.tolist()

    # Window
    win_start_d = phase_subs['mean_day'].min()
    win_end_d   = phase_subs['mean_day'].max()
    win_str     = f"Day {win_start_d:.0f} – Day {win_end_d:.0f} "\
                  f"(Mo {win_start_d/30:.1f} – Mo {win_end_d/30:.1f})"

    # Shopper penetration: % of all shoppers who bought any subclass in phase
    buyers = df_window[df_window['subclass'].isin(phase_sub_list)]['shopper_id'].nunique()
    pct    = buyers / df_window['shopper_id'].nunique() * 100

    # Top rules active in phase-adjacent cohort window
    phase_cohort = phase_subs['peak_cohort_idx'].mode()
    top_rule_str = "N/A"
    if not rules_df.empty and not phase_cohort.empty:
        pc_label = COHORT_LABELS[int(phase_cohort.iloc[0])] \
                   if int(phase_cohort.iloc[0]) < len(COHORT_LABELS) else None
        if pc_label:
            phase_rules = rules_df[rules_df['cohort'] == pc_label].sort_values(
                'lift', ascending=False
            )
            if not phase_rules.empty:
                top_rule_str = phase_rules.iloc[0]['rule']

    phase_summary_rows.append({
        'Lifecycle Phase'       : phase_name.strip(),
        'Time Window'           : win_str,
        'Dominant Subclasses'   : ', '.join(phase_sub_list[:5]),
        '# Subclasses'          : len(phase_sub_list),
        'Shopper Penetration %' : f"{pct:.1f}%",
        'Top Basket Rule'       : top_rule_str,
    })

phase_summary = pd.DataFrame(phase_summary_rows)
print("=== Baby Category Lifecycle Phases ===\n")
pd.set_option('display.max_colwidth', 80)
phase_summary

In [ ]:
# ── 9.2 Visual summary table ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(16, len(phase_summary) * 1.1 + 1.5))
ax.axis('off')

col_widths = [0.18, 0.16, 0.30, 0.08, 0.12, 0.22]
header_cols = list(phase_summary.columns)

table = ax.table(
    cellText=phase_summary.values,
    colLabels=header_cols,
    loc='center',
    cellLoc='left',
)
table.auto_set_font_size(False)
table.set_fontsize(8)
table.scale(1, 2.2)

# Header style
for j in range(len(header_cols)):
    table[(0, j)].set_facecolor('#2C3E50')
    table[(0, j)].set_text_props(color='white', fontweight='bold')

# Row colours per phase
for i, color in enumerate(PALETTE[:len(phase_summary)]):
    for j in range(len(header_cols)):
        table[(i + 1, j)].set_facecolor(color + '22')  # light tint

ax.set_title('Baby Category Lifecycle Phase Summary',
             fontsize=13, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('P2_07_lifecycle_phase_summary.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 10. Outputs

In [ ]:
# ── 10.1 Save all artefacts ────────────────────────────────────────────────────

# Subclass velocity table
velocity.to_csv('P2_subclass_velocity.csv', index=False)
print("Saved: P2_subclass_velocity.csv")

# All cohort MBA rules
if not rules_df.empty:
    rules_out = rules_df[['cohort', 'rule', 'antecedents_str', 'consequents_str',
                           'support', 'confidence', 'lift', 'n_shoppers']]
    rules_out.to_csv('P2_cohort_mba_rules.csv', index=False)
    print(f"Saved: P2_cohort_mba_rules.csv  ({len(rules_df):,} rules)")

# Global transition probabilities
hm_pivot.to_csv('P2_transition_matrix.csv')
print("Saved: P2_transition_matrix.csv")

# Subclass phase assignments
sc_features.reset_index().to_csv('P2_subclass_phase_assignments.csv', index=False)
print("Saved: P2_subclass_phase_assignments.csv")

# Lifecycle phase summary
phase_summary.to_csv('P2_lifecycle_phase_summary.csv', index=False)
print("Saved: P2_lifecycle_phase_summary.csv")

# T-Zero aligned transactions (window)
df_window[['shopper_id', 'date', 't_zero', 'days_since_zero',
           'month_since_zero', 'cohort', 'subclass',
           'total_sales', 'total_units']].to_parquet(
    'P2_tzero_aligned_transactions.parquet', index=False
)
print("Saved: P2_tzero_aligned_transactions.parquet")

print("\n=== Phase 2 (Temporal MBA) Complete ===")
print("Outputs ready for Phase 3 (Predictive Lifecycle Modeling):")
print("  • T-Zero aligned transaction file")
print("  • Subclass velocity + phase assignments")
print("  • Cohort-level association rules")
print("  • Transition probability matrix")
print("  • Lifecycle phase summary table")